[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/02_rebuild_atf_thresholds.ipynb)


In [1]:
# Colab / local repository setup
# Run this cell first when opening the notebook in Google Colab. It clones the
# repository, installs the pinned requirements, and makes data/ plus src/ imports
# available from the same execution context used by the local notebooks.
from pathlib import Path
import os
import subprocess
import sys


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


def _run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


REPO_URL = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
REPO_BRANCH = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
PROJECT_DIRNAME = os.environ.get("ASTROMODEL_PROJECT_DIRNAME", "astromodel_proving")

if _running_in_colab():
    project_root = Path("/content") / PROJECT_DIRNAME
    if not project_root.exists():
        clone_cmd = [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(project_root),
        ]
        try:
            _run(clone_cmd)
        except subprocess.CalledProcessError:
            # Some forks/default branches may not be named like REPO_BRANCH.
            # Retry without an explicit branch before surfacing the clone error.
            _run(["git", "clone", "--depth", "1", REPO_URL, str(project_root)])
    os.chdir(project_root)
    requirements = project_root / "requirements.txt"
    if requirements.exists():
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    project_root = next(
        (
            candidate
            for candidate in candidates
            if (candidate / "src").is_dir() and (candidate / "data").is_dir()
        ),
        current,
    )
    os.chdir(project_root)

os.environ["ASTROMODEL_PROJECT_ROOT"] = str(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"ASTROMODEL_PROJECT_ROOT={project_root}")
print(f"Working directory={Path.cwd()}")
print(f"data exists={(project_root / 'data').exists()}, src exists={(project_root / 'src').exists()}")


ASTROMODEL_PROJECT_ROOT=/home/xav/code/astromodel_proving
Working directory=/home/xav/code/astromodel_proving
data exists=True, src exists=True


# Step 02 — Rebuild region-aware ATF thresholds from the 37 files

This notebook reruns the ATF threshold rebuild on the **renamed 37-file dataset**.

## Reuse from `astro_atf_analysis_improved_sectioned.ipynb`

This step **does** reuse/adapt the working logic from the reference notebook through `src.atf_reference_adapter` and `src.atf_step02`:

- ATF discovery
- ATF parsing
- preprocessing / artifact handling
- sweep-level feature extraction

What is new here is the **repository-facing packaging** around that logic:

- region-aware threshold tables,
- redundancy diagnostics,
- region-effect summaries,
- feature reliability weights,
- and condition-level reliability.

## Important notebook display change

This notebook deliberately shows:
- the full canonical sweep-level feature table,
- the full redundancy table,
- the full region-effect table,
- and the full condition-level reliability table.

## Not included here

The Numba-vs-NumPy benchmark is intentionally omitted from step 02.  
That benchmark only becomes meaningful in later optimization-heavy steps.

In [2]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.atf_step02 import run_step02_rebuild_atf_thresholds

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT

PosixPath('/home/xav/code/astromodel_proving')

In [3]:
results = run_step02_rebuild_atf_thresholds(PROJECT_ROOT)

feature_df = results["feature_table_by_sweep"]
cell_counts = results["region_condition_cell_counts"]
redundancy = results["redundancy_diagnostics"]
reliability = results["feature_reliability_weights"]
condition_reliability = results["condition_feature_reliability"]
thresholds = results["condition_region_sweep_thresholds"]
region_effects = results["region_effect_summary"]

print("written outputs:", sorted((PROJECT_ROOT / "outputs" / "features").glob("*.csv")))
print("feature rows:", len(feature_df), "unique files:", feature_df["file_id"].nunique())

/home/xav/miniconda3/lib/python3.13/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


written outputs: [PosixPath('/home/xav/code/astromodel_proving/outputs/features/atf_region_condition_inventory.csv'), PosixPath('/home/xav/code/astromodel_proving/outputs/features/condition_feature_reliability.csv'), PosixPath('/home/xav/code/astromodel_proving/outputs/features/condition_region_sweep_thresholds.csv'), PosixPath('/home/xav/code/astromodel_proving/outputs/features/feature_correlation_summary.csv'), PosixPath('/home/xav/code/astromodel_proving/outputs/features/feature_reliability_weights.csv'), PosixPath('/home/xav/code/astromodel_proving/outputs/features/feature_table_by_sweep.csv'), PosixPath('/home/xav/code/astromodel_proving/outputs/features/global_pooled_sweep_thresholds.csv'), PosixPath('/home/xav/code/astromodel_proving/outputs/features/legacy_threshold_preview.csv'), PosixPath('/home/xav/code/astromodel_proving/outputs/features/performance_benchmark.csv'), PosixPath('/home/xav/code/astromodel_proving/outputs/features/preprocess_qc_by_sweep.csv'), PosixPath('/home/

## Region × condition cell counts

In [4]:
display(cell_counts)

fig, ax = plt.subplots(figsize=(7, 4))
plot_df = cell_counts.copy()
plot_df["label"] = plot_df["region"] + "_" + plot_df["condition"]
ax.bar(plot_df["label"], plot_df["n_cells"])
ax.set_ylabel("n_cells")
ax.set_title("ATF cell counts by region and condition")
ax.tick_params(axis="x", rotation=45)
plt.show()

,region,condition,n_cells,expected_n_cells,matches_expected,small_stratum
0,DH,CONTROL,7,7,True,False
1,DH,MFA,6,6,True,False
2,DH,MFA_BA,6,6,True,False
3,VH,CONTROL,4,4,True,True
4,VH,MFA,7,7,True,False
5,VH,MFA_BA,7,7,True,False


/tmp/ipykernel_262881/526010499.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Canonical sweep-level feature table

In [5]:
feature_view = feature_df[
    [
        "file_id",
        "region",
        "condition",
        "sweep",
        "peak_depolarization_mV",
        "stim_end_depolarization_mV",
        "rise_slope_mV_per_s",
        "rise_tau_s",
        "plateau_slope_mV_per_s",
        "decay_slope_mV_per_s",
        "decay_tau_s",
        "undershoot_magnitude_mV",
        "return_slope_mV_per_s",
        "n_artifact_points_total",
    ]
].copy()
print(feature_view.round(4).to_string(index=False))

       file_id region condition  sweep  peak_depolarization_mV  stim_end_depolarization_mV  rise_slope_mV_per_s  rise_tau_s  plateau_slope_mV_per_s  decay_slope_mV_per_s  decay_tau_s  undershoot_magnitude_mV  return_slope_mV_per_s  n_artifact_points_total
1_DH_1_CONTROL     DH   CONTROL      1                  7.2864                      6.7444               8.9886      0.2451                 -0.0340               10.6624       0.2732                   1.3220                 0.0323                        0
1_DH_1_CONTROL     DH   CONTROL      2                 10.6458                      9.9183              14.0461      0.2348                 -0.0318               14.5677       0.2906                   1.9103                 0.0411                        0
1_DH_1_CONTROL     DH   CONTROL      3                 14.0393                     13.2752              19.7089      0.2145                 -0.0108               18.4372       0.2857                   2.3144                 0.0476  

## Redundancy diagnostics

In [6]:
print(redundancy.round(4).to_string(index=False))

                 feature_a                  feature_b  n_complete_pairs  pearson_r  spearman_r  abs_spearman_r  redundant_flag
    peak_depolarization_mV stim_end_depolarization_mV               222     0.9986      0.9984          0.9984            True
       rise_slope_mV_per_s                 rise_tau_s               222    -0.6653     -0.9251          0.9251           False
       rise_slope_mV_per_s       decay_slope_mV_per_s               222     0.9196      0.8891          0.8891           False
                rise_tau_s       decay_slope_mV_per_s               222    -0.5775     -0.8253          0.8253           False
                rise_tau_s    undershoot_magnitude_mV               222    -0.6434     -0.8180          0.8180           False
      decay_slope_mV_per_s    undershoot_magnitude_mV               222     0.7795      0.7787          0.7787           False
       rise_slope_mV_per_s    undershoot_magnitude_mV               222     0.8231      0.7759          0.7759 

## Condition-level feature reliability

In [7]:
print(condition_reliability.round(4).to_string(index=False))

condition                    feature  n_regions  mean_missing_rate  mean_completeness  mean_reliability_weight  any_small_stratum
  CONTROL       decay_slope_mV_per_s          2             0.0000             1.0000                   0.9000               True
  CONTROL                decay_tau_s          2             0.0000             1.0000                   0.9000               True
  CONTROL     peak_depolarization_mV          2             0.0000             1.0000                   0.4500               True
  CONTROL     plateau_slope_mV_per_s          2             0.0000             1.0000                   0.9000               True
  CONTROL      return_slope_mV_per_s          2             0.0774             0.9226                   0.8310               True
  CONTROL        rise_slope_mV_per_s          2             0.0000             1.0000                   0.9000               True
  CONTROL                 rise_tau_s          2             0.0000             1.0000     

## Region-specific feature reliability

In [8]:
print(reliability.round(4).to_string(index=False))

region condition                    feature  n_rows  n_cells  n_non_missing  missing_rate  completeness  max_abs_spearman  redundant_flag  small_stratum  reliability_weight
    DH   CONTROL       decay_slope_mV_per_s      42        7             42        0.0000        1.0000            0.8891           False          False              1.0000
    DH   CONTROL                decay_tau_s      42        7             42        0.0000        1.0000            0.7735           False          False              1.0000
    DH   CONTROL     peak_depolarization_mV      42        7             42        0.0000        1.0000            0.9984            True          False              0.5000
    DH   CONTROL     plateau_slope_mV_per_s      42        7             42        0.0000        1.0000            0.7360           False          False              1.0000
    DH   CONTROL      return_slope_mV_per_s      42        7             39        0.0714        0.9286            0.4825           Fal

## Region-aware thresholds

In [9]:
print(thresholds.round(4).to_string(index=False))

condition region  sweep                    feature  n_total_rows  n_non_missing  missing_rate  median      q1      q3     iqr  acceptable_lower  acceptable_upper threshold_scope  reliability_weight
  CONTROL     DH      1       decay_slope_mV_per_s             7              7        0.0000 10.6624  6.6801 11.7468  5.0667           -0.9200           19.3469 region_specific              1.0000
  CONTROL     DH      1                decay_tau_s             7              7        0.0000  0.2648  0.2296  0.3802  0.1507            0.0036            0.6062 region_specific              1.0000
  CONTROL     DH      1     peak_depolarization_mV             7              7        0.0000  7.6025  6.8372  7.8881  1.0509            5.2609            9.4644 region_specific              0.5000
  CONTROL     DH      1     plateau_slope_mV_per_s             7              7        0.0000 -0.0208 -0.0284  0.0006  0.0290           -0.0719            0.0441 region_specific              1.0000
  CONTROL 

## Region-effect summary

In [10]:
print(region_effects.round(4).to_string(index=False))

condition  sweep                    feature  n_dh  n_vh  median_dh  median_vh  vh_minus_dh_median  bootstrap_ci_low  bootstrap_ci_high  small_stratum
  CONTROL      1       decay_slope_mV_per_s     7     4    10.6624     5.8731             -4.7893           -6.6425             2.0408           True
  CONTROL      2       decay_slope_mV_per_s     7     4    14.5677     8.3477             -6.2200          -10.2377             2.9844           True
  CONTROL      3       decay_slope_mV_per_s     7     4    18.4372    11.5634             -6.8738          -12.3879             6.1643           True
  CONTROL      4       decay_slope_mV_per_s     7     4    22.5288    12.3095            -10.2193          -13.3983             4.6106           True
  CONTROL      5       decay_slope_mV_per_s     7     4    24.8577    15.3603             -9.4973          -14.7422             8.5924           True
  CONTROL      6       decay_slope_mV_per_s     7     4    27.3451    14.9625            -12.3827   

## Consequence for later steps

Two practical conclusions stand out:

1. `peak_depolarization_mV` and `stim_end_depolarization_mV` are near-duplicates here and should not both dominate later losses.
2. `return_slope_mV_per_s` has condition-dependent missingness, so its weight should be explicitly moderated rather than treated as uniformly reliable across conditions.

## Post-execution scientific status

Executed status for reviewer response: Step 02 supports R2/R6/R7 by rebuilding full target-scope ATF feature tables and region-aware thresholds from 222 sweeps (37 cells x 6 currents). It records 432 region-specific, 216 region-pooled, and 72 global pooled threshold rows. These thresholds define the objective acceptance and validation contract used downstream; they do not by themselves establish mechanism classes or degeneracy.